In [ ]:
import os







import sys







try:







    from google.colab import drive







    if not os.path.exists('/content/drive'):







        drive.mount('/content/drive')







    REPO_ROOT = '/content/drive/MyDrive/Stocks'







except ImportError:







    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))















if REPO_ROOT not in sys.path:







    sys.path.insert(0, REPO_ROOT)















import yfinance as yf







from IPython.display import HTML, display















from portfolio.allocations import (







    TARGET_WEIGHTS, MONTHLY_DEPOSIT,







    NUCLEAR_BASKET_TARGETS, QUANTUM_BASKET_TARGETS, CYBER_BASKET_TARGETS,







    INDUSTRIAL_BASKET_TARGETS, SPECGROWTH_BASKET_TARGETS,







    my_current_shares, ETF_LOOK_THROUGH,







)







from portfolio.crypto import (







    CRYPTO_TARGET_WEIGHTS, my_current_crypto, MONTHLY_DEPOSIT_CHF,







)







from portfolio.helpers import get_price, get_live_fx_rate















# =========================================================================







# 1. FETCH LIVE PRICES







# =========================================================================







print('Fetching stock/ETF prices...')







stock_prices = {t: get_price(t) for t in my_current_shares.keys()}















print('Fetching crypto prices...')







crypto_prices_usd = {}







for ticker in CRYPTO_TARGET_WEIGHTS:







    try:







        data = yf.Ticker(ticker).history(period='1d')







        crypto_prices_usd[ticker] = float(data['Close'].iloc[-1]) if not data.empty else 0.01







    except Exception:







        crypto_prices_usd[ticker] = 0.01















print('Fetching FX rates...')







USD_TO_CHF = get_live_fx_rate('USD', 'CHF')







EUR_TO_CHF = get_live_fx_rate('EUR', 'CHF')















# =========================================================================







# 2. COMPUTE PORTFOLIO VALUES







# =========================================================================







stock_values = {t: shares * stock_prices.get(t, 0) for t, shares in my_current_shares.items()}















nuclear_val = sum(stock_values.get(t, 0) for t in NUCLEAR_BASKET_TARGETS)







quantum_val = sum(stock_values.get(t, 0) for t in QUANTUM_BASKET_TARGETS)







cyber_val = sum(stock_values.get(t, 0) for t in CYBER_BASKET_TARGETS)







industrial_val = sum(stock_values.get(t, 0) for t in INDUSTRIAL_BASKET_TARGETS)







specgrowth_val = sum(stock_values.get(t, 0) for t in SPECGROWTH_BASKET_TARGETS)















macro_values = {







    'XAIX.DE': stock_values.get('XAIX.DE', 0),







    'SMHV.SW': stock_values.get('SMHV.SW', 0),







    'QDVE.DE': stock_values.get('QDVE.DE', 0),







    'NUCLEAR_SATELLITE': nuclear_val,







    'QUANTUM_SATELLITE': quantum_val,







    'CYBER_SATELLITE': cyber_val,







    'INDUSTRIAL_SATELLITE': industrial_val,







    'SPECGROWTH_SATELLITE': specgrowth_val,







}







total_stock_val = sum(macro_values.values())















crypto_values_usd = {







    c: my_current_crypto.get(c, 0) * crypto_prices_usd.get(c, 0)







    for c in CRYPTO_TARGET_WEIGHTS







}







total_crypto_usd = sum(crypto_values_usd.values())







total_crypto_chf = total_crypto_usd * USD_TO_CHF







total_portfolio_usd = total_stock_val + total_crypto_usd















# =========================================================================







# 3. ETF LOOK-THROUGH EXPOSURE







# =========================================================================







exposure = {}







exposure_sources = {}















for etf, holdings in ETF_LOOK_THROUGH.items():







    macro_weight = TARGET_WEIGHTS.get(etf, 0)







    for stock, sub_weight in holdings.items():







        net = macro_weight * sub_weight







        exposure[stock] = exposure.get(stock, 0) + net







        exposure_sources.setdefault(stock, []).append(etf)















for stock, sub_weight in NUCLEAR_BASKET_TARGETS.items():







    net = TARGET_WEIGHTS['NUCLEAR_SATELLITE'] * sub_weight







    exposure[stock] = exposure.get(stock, 0) + net







    exposure_sources.setdefault(stock, []).append('NUCLEAR')















for stock, sub_weight in QUANTUM_BASKET_TARGETS.items():







    net = TARGET_WEIGHTS['QUANTUM_SATELLITE'] * sub_weight







    exposure[stock] = exposure.get(stock, 0) + net







    exposure_sources.setdefault(stock, []).append('QUANTUM')















for stock, sub_weight in CYBER_BASKET_TARGETS.items():







    net = TARGET_WEIGHTS['CYBER_SATELLITE'] * sub_weight







    exposure[stock] = exposure.get(stock, 0) + net







    exposure_sources.setdefault(stock, []).append('CYBER')















for stock, sub_weight in INDUSTRIAL_BASKET_TARGETS.items():







    net = TARGET_WEIGHTS['INDUSTRIAL_SATELLITE'] * sub_weight







    exposure[stock] = exposure.get(stock, 0) + net







    exposure_sources.setdefault(stock, []).append('INDUSTRIAL')















for stock, sub_weight in SPECGROWTH_BASKET_TARGETS.items():







    net = TARGET_WEIGHTS['SPECGROWTH_SATELLITE'] * sub_weight







    exposure[stock] = exposure.get(stock, 0) + net







    exposure_sources.setdefault(stock, []).append('SPECGROWTH')















sorted_exposure = sorted(exposure.items(), key=lambda x: x[1], reverse=True)















# =========================================================================







# 4. HELPERS







# =========================================================================







def fmt_money(val):







    if abs(val) >= 1e6: return f'${val/1e6:,.2f}M'







    elif abs(val) >= 1e3: return f'${val:,.0f}'







    return f'${val:,.2f}'















def drift_cell(current_pct, target_pct):







    diff = current_pct - target_pct







    if abs(diff) < 2: color = '#1B5E20'







    elif abs(diff) < 5: color = '#E65100'







    else: color = '#B71C1C'







    return f'<td style="color:{color}; font-weight:bold;">{diff:+.1f}%</td>'















MACRO_LABELS = {







    'XAIX.DE': 'XAIX.DE — AI & Big Data Index',







    'SMHV.SW': 'SMHV.SW — Semiconductors',







    'QDVE.DE': 'QDVE.DE — S&P 500 Info Tech',







    'NUCLEAR_SATELLITE': 'Nuclear Satellite (CCJ, GEV, SRUUF, LEU, SMR, OKLO)',







    'QUANTUM_SATELLITE': 'Quantum Satellite (IONQ, QNT, QBTS, RGTI, QUBT)',







    'CYBER_SATELLITE': 'Cyber Satellite (CRWD, PANW)',







    'INDUSTRIAL_SATELLITE': 'Industrial Satellite (BWXT, POWL, VRT, FIX)',







    'SPECGROWTH_SATELLITE': 'SpecGrowth Satellite (RKLB, LSCC, CRDO, VKTX)',







}







MACRO_COLORS = {







    'XAIX.DE': '#E3F2FD', 'SMHV.SW': '#E3F2FD',







    'QDVE.DE': '#E3F2FD', 'NUCLEAR_SATELLITE': '#FFF8DC',







    'QUANTUM_SATELLITE': '#F3E6F5', 'CYBER_SATELLITE': '#FFEBEE',







    'INDUSTRIAL_SATELLITE': '#E8EAF6', 'SPECGROWTH_SATELLITE': '#E0F7FA',







}







CRYPTO_COLORS = {







    'BTC-USD': '#FFF3E0', 'ETH-USD': '#E3F2FD', 'SOL-USD': '#F3E6F5',







    'RENDER-USD': '#E8F5E9', 'LINK-USD': '#E3F2FD', 'XRP-USD': '#ECEFF1',







}







CRYPTO_LABELS = {







    'BTC-USD': 'Bitcoin', 'ETH-USD': 'Ethereum', 'SOL-USD': 'Solana',







    'RENDER-USD': 'Render', 'LINK-USD': 'Chainlink', 'XRP-USD': 'XRP',







}







SOURCE_COLORS = {







    'SMHV.SW': '#E6F0FA', 'QDVE.DE': '#E6F7F9', 'XAIX.DE': '#E6F4EA',







    'NUCLEAR': '#FCF7E6', 'QUANTUM': '#FAE6FA', 'CYBER': '#FCE8E6',







    'INDUSTRIAL': '#E8EAF6', 'SPECGROWTH': '#E0F7FA',







}







SOURCE_LABELS = {







    'SMHV.SW': 'Core Semiconductors (SMHV.SW)', 'QDVE.DE': 'S&P 500 Info Tech (QDVE.DE)',







    'XAIX.DE': 'AI & Big Data Index (XAIX.DE)',







    'NUCLEAR': 'Satellite Layer (NUCLEAR)', 'QUANTUM': 'Satellite Layer (QUANTUM)',







    'CYBER': 'Satellite Layer (CYBER)',







    'INDUSTRIAL': 'Satellite Layer (INDUSTRIAL)', 'SPECGROWTH': 'Satellite Layer (SPECGROWTH)',







}















# =========================================================================







# 5. BUILD HTML







# =========================================================================







html = []















# --- CSS ---







html.append("""<style>







.po-tab { overflow: hidden; border: 1px solid #000; background-color: #f1f1f1; margin-top: 10px; }







.po-tab button { background-color: inherit; float: left; border: none; outline: none; cursor: pointer; padding: 14px 16px; transition: 0.3s; color: black; font-weight: bold; font-size: 14px; }







.po-tab button:hover { background-color: #ddd; }







.po-tab button.active { background-color: #ccc; }







.po-tabcontent { display: none; padding: 20px; border: 1px solid #000; border-top: none; background: white; }







.po-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 13px; }







.po-table th { background: #2C3E50; color: white; padding: 8px 12px; text-align: left; font-weight: bold; }







.po-table td { padding: 6px 12px; border-bottom: 1px solid #ddd; color: #1a1a1a; }







.po-table tr:hover { filter: brightness(0.95); }







.po-total td { background: #2C3E50 !important; color: white !important; font-weight: bold; }







.po-header { font-size: 16px; font-weight: bold; margin-bottom: 12px; color: #1a1a1a; }







.po-summary { display: flex; gap: 20px; margin-bottom: 20px; flex-wrap: wrap; }







.po-card { background: white; padding: 16px 24px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); text-align: center; min-width: 180px; }







.po-card-value { font-size: 24px; font-weight: bold; color: #2C3E50; }







.po-card-label { font-size: 12px; color: #555; margin-top: 4px; }







</style>







<script>







function poOpenTab(evt, tabName) {







  var i, tc, tl;







  tc = document.getElementsByClassName('po-tabcontent');







  for (i = 0; i < tc.length; i++) { tc[i].style.display = 'none'; }







  tl = document.getElementsByClassName('po-tablink');







  for (i = 0; i < tl.length; i++) { tl[i].className = tl[i].className.replace(' active', ''); }







  document.getElementById(tabName).style.display = 'block';







  evt.currentTarget.className += ' active';







}







</script>""")















# --- Summary cards ---







stock_pct = total_stock_val / total_portfolio_usd * 100 if total_portfolio_usd > 0 else 0







crypto_pct = total_crypto_usd / total_portfolio_usd * 100 if total_portfolio_usd > 0 else 0















html.append('<div class="po-summary">')







html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_portfolio_usd)}</div><div class="po-card-label">Total Portfolio (approx USD)</div></div>')







html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_stock_val)}</div><div class="po-card-label">Stocks & ETFs ({stock_pct:.0f}%)</div></div>')







html.append(f'<div class="po-card"><div class="po-card-value">{fmt_money(total_crypto_usd)}</div><div class="po-card-label">Crypto ({crypto_pct:.0f}%)</div></div>')







html.append(f'<div class="po-card"><div class="po-card-value">{USD_TO_CHF:.4f}</div><div class="po-card-label">USD/CHF</div></div>')







html.append(f'<div class="po-card"><div class="po-card-value">{EUR_TO_CHF:.4f}</div><div class="po-card-label">EUR/CHF</div></div>')







html.append('</div>')















# --- Tab bar ---







html.append('<div class="po-tab">')







html.append('<button class="po-tablink active" onclick="poOpenTab(event, \'po_macro\')">Macro Allocation</button>')







html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_baskets\')">Satellite Baskets</button>')







html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_exposure\')">Stock Exposure</button>')







html.append('<button class="po-tablink" onclick="poOpenTab(event, \'po_crypto\')">Crypto</button>')







html.append('</div>')















# =========================================================================







# TAB 1: MACRO ALLOCATION







# =========================================================================







html.append('<div id="po_macro" class="po-tabcontent" style="display:block;">')







html.append('<div class="po-header">Stocks & ETFs — Macro Allocation</div>')







html.append('<table class="po-table">')







html.append('<tr><th>Asset</th><th>Value</th><th>Current %</th><th>Target %</th><th>Drift</th></tr>')















for asset, target_pct in TARGET_WEIGHTS.items():







    val = macro_values.get(asset, 0)







    cur_pct = val / total_stock_val * 100 if total_stock_val > 0 else 0







    tgt_pct = target_pct * 100







    label = MACRO_LABELS.get(asset, asset)







    bg = MACRO_COLORS.get(asset, '#FFFFFF')







    yf_ticker = asset.replace("_SATELLITE", "")







    if asset in ("XAIX.DE", "SMHV.SW", "QDVE.DE"):







        yf_link = f'<a href="https://finance.yahoo.com/quote/{asset}/" target="_blank" style="color:#1565C0;">{label}</a>'







    else:







        yf_link = label







    html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b></td><td>{fmt_money(val)}</td><td>{cur_pct:.1f}%</td><td>{tgt_pct:.0f}%</td>{drift_cell(cur_pct, tgt_pct)}</tr>')















html.append(f'<tr class="po-total"><td>TOTAL</td><td>{fmt_money(total_stock_val)}</td><td>100%</td><td>100%</td><td>—</td></tr>')







html.append('</table>')















if total_stock_val > 0:







    chosen_stock = max(TARGET_WEIGHTS, key=lambda a: TARGET_WEIGHTS[a] - (macro_values.get(a, 0) / total_stock_val))







    html.append(f'<p style="margin-top:12px; font-size:14px; color:#1a1a1a;"><b>Monthly Buy:</b> {MACRO_LABELS.get(chosen_stock, chosen_stock)} — <b>\u20ac{MONTHLY_DEPOSIT:,.0f}</b></p>')















html.append('</div>')















# =========================================================================







# TAB 2: SATELLITE BASKETS







# =========================================================================







html.append('<div id="po_baskets" class="po-tabcontent">')







html.append('<div class="po-header">Satellite Baskets — Individual Holdings</div>')







html.append('<table class="po-table">')







html.append('<tr><th>Ticker</th><th>Basket</th><th>Price</th><th>Value</th><th>Basket %</th><th>Target %</th><th>Drift</th></tr>')















baskets = [







    ('Nuclear', NUCLEAR_BASKET_TARGETS, nuclear_val),







    ('Quantum', QUANTUM_BASKET_TARGETS, quantum_val),







    ('Cyber', CYBER_BASKET_TARGETS, cyber_val),







    ('Industrial', INDUSTRIAL_BASKET_TARGETS, industrial_val),







    ('SpecGrowth', SPECGROWTH_BASKET_TARGETS, specgrowth_val),







]







basket_colors = {'Nuclear': '#FFF8DC', 'Quantum': '#F3E6F5', 'Cyber': '#FFEBEE', 'Industrial': '#E8EAF6', 'SpecGrowth': '#E0F7FA'}















for basket_name, targets, basket_total in baskets:







    for ticker, target in targets.items():







        shares = my_current_shares.get(ticker, 0)







        price = stock_prices.get(ticker, 0)







        val = stock_values.get(ticker, 0)







        cur_pct = val / basket_total * 100 if basket_total > 0 else 0







        tgt_pct = target * 100







        bg = basket_colors.get(basket_name, '#FFFFFF')







        yf_link = f'<a href="https://finance.yahoo.com/quote/{ticker}/" target="_blank" style="color:#1565C0;">{ticker}</a>'







        html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b></td><td>{basket_name}</td><td>${price:,.2f}</td><td>{fmt_money(val)}</td><td>{cur_pct:.1f}%</td><td>{tgt_pct:.0f}%</td>{drift_cell(cur_pct, tgt_pct)}</tr>')















html.append('</table></div>')















# =========================================================================







# TAB 3: STOCK EXPOSURE







# =========================================================================







html.append('<div id="po_exposure" class="po-tabcontent">')







html.append('<div class="po-header">Underlying Stock Exposure — ETF Look-Through</div>')







html.append('<table class="po-table">')







html.append('<tr><th>Underlying Stock / Asset</th><th>Net Portfolio Weight</th><th>Primary Origin</th></tr>')















for stock, weight in sorted_exposure:







    sources = list(set(exposure_sources.get(stock, [])))







    weight_str = f'{weight * 100:.2f}%'







    if len(sources) > 1:







        source_label = f'Cross-ETF Overlap ({", ".join(sorted(sources))})'







        bg = '#F3E6FA'







    else:







        src = sources[0]







        source_label = SOURCE_LABELS.get(src, src)







        bg = SOURCE_COLORS.get(src, '#FFFFFF')







    yf_link = f'<a href="https://finance.yahoo.com/quote/{stock}/" target="_blank" style="color:#1565C0;">{stock}</a>'







    html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b></td><td>{weight_str}</td><td>{source_label}</td></tr>')















html.append('</table></div>')















# =========================================================================







# TAB 4: CRYPTO







# =========================================================================







html.append('<div id="po_crypto" class="po-tabcontent">')







html.append(f'<div class="po-header">Crypto Allocation (1 USD = {USD_TO_CHF:.4f} CHF)</div>')







html.append('<table class="po-table">')







html.append('<tr><th>Asset</th><th>Holdings</th><th>Price (USD)</th><th>Value (USD)</th><th>Value (CHF)</th><th>Current %</th><th>Target %</th><th>Drift</th></tr>')















for ticker, target_pct in CRYPTO_TARGET_WEIGHTS.items():







    holdings = my_current_crypto.get(ticker, 0)







    price = crypto_prices_usd.get(ticker, 0)







    val_usd = crypto_values_usd.get(ticker, 0)







    val_chf = val_usd * USD_TO_CHF







    cur_pct = val_usd / total_crypto_usd * 100 if total_crypto_usd > 0 else 0







    tgt_pct = target_pct * 100







    label = CRYPTO_LABELS.get(ticker, ticker)







    bg = CRYPTO_COLORS.get(ticker, '#FFFFFF')







    yf_link = f'<a href="https://finance.yahoo.com/quote/{ticker}/" target="_blank" style="color:#1565C0;">{ticker}</a>'







    html.append(f'<tr style="background:{bg};"><td><b>{yf_link}</b> ({label})</td><td>{holdings:,.4f}</td><td>${price:,.2f}</td><td>{fmt_money(val_usd)}</td><td>{val_chf:,.0f} CHF</td><td>{cur_pct:.1f}%</td><td>{tgt_pct:.0f}%</td>{drift_cell(cur_pct, tgt_pct)}</tr>')















html.append(f'<tr class="po-total"><td>TOTAL</td><td></td><td></td><td>{fmt_money(total_crypto_usd)}</td><td>{total_crypto_chf:,.0f} CHF</td><td>100%</td><td>100%</td><td>—</td></tr>')







html.append('</table>')















if total_crypto_usd > 0:







    chosen = max(CRYPTO_TARGET_WEIGHTS, key=lambda c: CRYPTO_TARGET_WEIGHTS[c] - (crypto_values_usd.get(c, 0) / total_crypto_usd))







    price_chf = crypto_prices_usd[chosen] * USD_TO_CHF







    units = MONTHLY_DEPOSIT_CHF / price_chf if price_chf > 0 else 0







    html.append(f'<p style="margin-top:12px; font-size:14px; color:#1a1a1a;"><b>Crypto Buy:</b> {chosen} ({CRYPTO_LABELS.get(chosen, "")}) — {units:.4f} tokens at {price_chf:,.2f} CHF = <b>{MONTHLY_DEPOSIT_CHF:,.0f} CHF</b></p>')















html.append('</div>')















# =========================================================================







# 6. RENDER







# =========================================================================







display(HTML('\n'.join(html)))







print('\nPortfolio overview rendered.')































































